# Stage 2 — Conditional VAE + Physics Constraints

This notebook extends the rule-based STORM-like simulator with generative modeling. It compares:

1. **CVAE** — statistical/data-driven baseline.
2. **Physics-CVAE (soft constraints)** — adds penalties for energy balance, non-negative consumption/PV, line capacity, and voltage bounds.
3. **Physics-CVAE (hard projection)** — projects generated samples onto exact `net_load = consumption - PV` and non-negativity constraints.

The goal is not to reproduce the full SHARE PhD project. It is a compact experiment showing how statistical fidelity and physical plausibility can be evaluated together.

## 1. Imports

Keep `energy_simulator_stage2.py` and `physics_cvae.py` in the same folder as this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

from stage1_simulator.energy_simulator import SimulationConfig, run as run_simulator
from stage2_physics_cvae.physics_cvae import (
    CVAEConfig, ConditionalVAE, prepare_data, train_model, evaluate_and_save
)
import torch

DATA_DIR = Path("stage2_data")
RESULT_DIR = Path("stage2_cvae_results")

## 2. Generate a larger physics-aware synthetic dataset

Seven or fourteen days are too few for a generative model. For the mini-project, generate at least **180 days**. You can increase this to 365 later.

In [ ]:
if not (DATA_DIR / "consumer_timeseries.csv").exists():
    sim_config = SimulationConfig(
        seed=42,
        days=180,
        interval_minutes=15,
        n_consumers=18,
        start="2026-01-01 00:00:00+00:00",
    )
    run_simulator(sim_config, DATA_DIR)
else:
    print("Using existing simulator output:", DATA_DIR.resolve())

## 3. Prepare daily multivariate samples

Each day contains 96 time steps. The model jointly generates three channels for every consumer:

- consumption,
- PV generation,
- net load.

The conditional variables include weekend/day-of-week information and daily weather summaries.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_cfg = CVAEConfig(
    latent_dim=24,
    hidden_dim=256,
    batch_size=32,
    epochs=100,
    learning_rate=1e-3,
    beta_kl=1e-3,
    physics_weight=10.0,
    test_fraction=0.2,
    seed=42,
)

data = prepare_data(DATA_DIR, train_cfg, device)
print("device:", device)
print("train days:", len(data.x_train))
print("test days:", len(data.x_test))
print("consumers:", len(data.consumer_ids))
print("input dimension:", data.x_train.shape[1])

## 4. Build the two trainable models

Both models start from the same initial weights so the comparison is fair.

In [ ]:
x_dim = data.x_train.shape[1]
c_dim = data.c_train.shape[1]

baseline_cvae = ConditionalVAE(
    x_dim=x_dim, c_dim=c_dim,
    latent_dim=train_cfg.latent_dim, hidden_dim=train_cfg.hidden_dim
).to(device)

physics_cvae = ConditionalVAE(
    x_dim=x_dim, c_dim=c_dim,
    latent_dim=train_cfg.latent_dim, hidden_dim=train_cfg.hidden_dim
).to(device)

physics_cvae.load_state_dict(baseline_cvae.state_dict())

## 5. Train baseline CVAE

In [ ]:
baseline_history = train_model(
    baseline_cvae, data, train_cfg, physics_aware=False
)
baseline_history.tail()

## 6. Train Physics-CVAE with soft constraints

The physics loss penalizes:

- mismatch in `net_load = consumption - PV`,
- negative consumption or PV generation,
- line overloads,
- voltage-limit violations.

The loss is applied both to reconstructions and to samples drawn from the prior.

In [ ]:
physics_history = train_model(
    physics_cvae, data, train_cfg, physics_aware=True
)
physics_history.tail()

## 7. Compare statistical fidelity and physical consistency

The evaluation compares real simulator data, the baseline CVAE, the soft Physics-CVAE, and a hard-projected Physics-CVAE.

In [ ]:
metrics = evaluate_and_save(
    RESULT_DIR, data,
    baseline_cvae, physics_cvae,
    baseline_history, physics_history,
    seed=142,
)
display(metrics)

### How to interpret the metrics

A good generative model should not be judged by a single number. Look for a trade-off:

- **Mean/std profile RMSE:** statistical similarity of daily load shapes.
- **Daily energy / peak-load gaps:** whether generated days have realistic scale.
- **Lag-1 autocorrelation:** whether temporal smoothness is preserved.
- **Power-balance RMSE:** physical consistency between consumption, PV and net load.
- **Overload / voltage-violation fraction:** network feasibility.

The hard projection should make the energy-balance identity exact, but it may worsen statistical fidelity. That trade-off is itself an interesting result.

In [ ]:
display(Image(filename=str(RESULT_DIR / "mean_daily_profile_comparison.png")))
display(Image(filename=str(RESULT_DIR / "physical_consistency_comparison.png")))

## 8. Next extensions

Keep these as future work rather than trying to implement everything at once:

- replace CVAE with diffusion / flow matching,
- represent the network explicitly with a GNN,
- use a proper AC or linearized power-flow solver,
- add phase imbalance constraints,
- add differential privacy,
- evaluate train-synthetic-test-real forecasting.

For the PhD application, the important point is that this mini-project already demonstrates the full reasoning loop: **synthetic data → generative model → physical constraints → statistical and physical evaluation**.